In [ ]:
import pandas as pd
import numpy as np
# Read the Excel file
df=pd.read_csv("E:/STUDIES.AI/csv_files/project/Uber_Eats_data.csv")
df=df.drop(columns=["phone"])
print(df.shape)

In [ ]:
df['rate'] = df['rate'].replace('NEW', np.nan)
# Check for missing values
print(df.isnull().sum())
df=df.dropna()

In [ ]:
df['rate'] = df['rate'].str.split('/').str[0].astype(float)
# Checking Duplicates
print(df.duplicated().sum())
#Removing Duplicates
df=df.drop_duplicates()
print(df.shape)

In [ ]:
# Converting cost to float
df["approx_cost(for two people)"]=(df["approx_cost(for two people)"].str.replace(",","",regex=False).astype(float))
print(df["approx_cost(for two people)"].dtype)

In [ ]:
df["name"] = (df["name"]                    
    .str.replace(r"[^a-zA-Z0-9&,'\-@!. ]", "", regex=True).str.strip().str.title())
#Renaming columns
df = df.rename(columns={
     "name": "restaurant_name",
     "rate": "rating",
     "approx_cost(for two people)": "approx_cost(for_two_people)"
 })
print(df.head())


In [ ]:
def rating_category(rating):
    if rating < 3.5:
        return "Poor"
    elif rating < 4.0:
        return "Average"
    elif rating < 4.5:
        return "Good"
    else:
        return "Excellent"
df["rating_category"] = df["rating"].apply(rating_category)
print(df["rating_category"].value_counts())    

In [ ]:
print(df["approx_cost(for_two_people)"].describe())

In [ ]:
def cost_category(approx_cost):
    if approx_cost <= 500:
        return "Low"
    elif approx_cost <= 1000:
        return "Medium"
    elif approx_cost <= 2000:
        return "High"
    else:
        return "Premium"

df["cost_category"] = df["approx_cost(for_two_people)"].apply(cost_category)
print(df.head())
#df.to_csv("E:/STUDIES.AI/csv_files/project/Uber_final_data.csv",index=False)

In [9]:
import json
with open("E:/STUDIES.AI/csv_files/project/orders.json") as file:
    data = json.load(file)
data = pd.DataFrame(data)
#data.to_csv("E:/STUDIES.AI/csv_files/project/orders1.csv",index=False)

In [ ]:
df1=pd.read_csv("E:/STUDIES.AI/csv_files/project/orders1.csv")
print(df1.isnull().sum())
print(df.duplicated().sum())

In [ ]:
df1["restaurant_name"]=df1["restaurant_name"].str.replace(r"[^a-zA-Z0-9&,'\-@!. ]", "", regex=True).str.strip().str.title()
print(df1.head())

In [ ]:
df1["order_date"] = pd.to_datetime(df1["order_date"])
print(df1["order_date"].dtype)
print(df1["order_date"].head())

In [ ]:
#Changing order_value to float
df1["order_value"] = df1["order_value"].astype(float)
print(df1["order_value"].dtype)
print(df1["order_value"].describe())
#df1.to_csv("E:/STUDIES.AI/csv_files/project/orders_final.csv",index=False)

Creating SQL Connection Using SQLite3

In [14]:
import pandas as pd
import sqlite3
df1 = pd.read_csv("E:/STUDIES.AI/csv_files/project/Uber_final_data.csv")
df2 = pd.read_csv("E:/STUDIES.AI/csv_files/project/orders_final.csv")
conn = sqlite3.connect("uberbase.db")
cursor = conn.cursor() 

In [15]:
# Create Restaurant table
cursor.execute("""
CREATE TABLE IF NOT EXISTS Restaurant (
    restaurant_name TEXT,
    online_order TEXT,
    book_table TEXT,
    rating FLOAT,
    votes INTEGER,
    location TEXT,
    rest_type TEXT,
    dish_liked TEXT,
    cuisines TEXT,
    approx_cost_for_two_people FLOAT,
    listed_in_type TEXT,
    listed_in_city TEXT,
    rating_category TEXT,
    cost_category TEXT
);
""")
conn.commit()

In [16]:
cursor.execute("DELETE FROM Restaurant")
data = df1.values.tolist()
# Insert values
cursor.executemany("""
INSERT INTO  Restaurant( 
restaurant_name, online_order, book_table, rating, votes,
location, rest_type, dish_liked, cuisines,
approx_cost_for_two_people, listed_in_type, listed_in_city,rating_category,cost_category
) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", data)
conn.commit()

In [ ]:
#query1
query1="""
SELECT location, ROUND(AVG(rating),1) AS average_rating, COUNT(*) AS total_restaurants
FROM Restaurant
GROUP BY location
ORDER BY average_rating DESC
LIMIT 5;
"""
cursor.execute(query1)
data = cursor.fetchall()
df = pd.DataFrame(
    data,
    columns=['location', 'average_rating', 'total_restaurants']
)
print("\nBangalore locations have the highest average restaurant ratings(query1)\n",df)

In [ ]:
#Query2
Query2="""
SELECT location,
       COUNT(*) AS total_restaurants
FROM Restaurant
GROUP BY location
ORDER BY total_restaurants DESC
LIMIT 10
;
"""
cursor.execute(Query2)
data = cursor.fetchall()
df = pd.DataFrame(data,columns=["Location", "Total Restaurants"])
print("\nTop 10 Over-Saturated Locations(query 2):\n")
print(df)

In [ ]:
#query3
query3 = """
SELECT online_order,
       ROUND(AVG(rating), 2) AS average_rating
       FROM Restaurant
GROUP BY online_order;
"""
cursor.execute(query3)
data = cursor.fetchall()
df = pd.DataFrame(
    data,
    columns=["Online Order", "Average Rating"]
)
print("\nDoes online ordering improve restaurant ratings(query3)\n",df)

In [ ]:
#quer4
query4 = """
SELECT
    book_table,
    ROUND(AVG(rating), 2) AS average_rating
FROM Restaurant
GROUP BY book_table
ORDER BY average_rating DESC;
"""
cursor.execute(query4)
data = cursor.fetchall()
df = pd.DataFrame(
    data,
    columns=["Table Booking", "Average Rating"]
)

print("\nDoes table booking correlate with higher customer ratings(query4)\n",df)

In [ ]:
#query5
query5 = """
SELECT
      CASE
          WHEN approx_cost_for_two_people<=500 THEN "LOW"
          WHEN approx_cost_for_two_people<=1000 THEN "MEDIUM"
          WHEN approx_cost_for_two_people<=2000 THEN "HIGH"
          ELSE "PREMIUM"
      END AS price_range,    
      ROUND(AVG(rating),2) as average_rating,
      COUNT(*) AS total_restaurants
FROM Restaurant
GROUP BY price_range
ORDER BY average_rating DESC;"""

cursor.execute(query5)
data = cursor.fetchall()
df = pd.DataFrame(
    data,
    columns=["price_range", "Average_rating","Total_Restaurants"]
)
print("\nWhat price range delivers the best customer satisfaction(query5)\n",df)

In [ ]:
#query6
query6 = """
SELECT
      CASE
          WHEN approx_cost_for_two_people<=500 THEN "LOW"
          WHEN approx_cost_for_two_people<=1000 THEN "MEDIUM"
          WHEN approx_cost_for_two_people<=2000 THEN "HIGH"
          ELSE "PREMIUM"
      END AS price_range,    
      ROUND(AVG(rating),2) as average_rating,
      COUNT(*) AS total_restaurants
FROM Restaurant
GROUP BY price_range
ORDER BY average_rating DESC;"""

cursor.execute(query6)
data = cursor.fetchall()
df = pd.DataFrame(
    data,
    columns=["price_range", "Average_rating","Total_Restaurants"]
)
print("\nHow do low, mid, and premium-priced restaurants perform in terms of ratings?(query6)\n",df)

In [ ]:
#Query7
query7="""
SELECT cuisines, COUNT(*) AS total_restaurants
FROM Restaurant
GROUP BY cuisines
ORDER BY total_restaurants DESC
LIMIT 5;
"""
cursor.execute(query7)
data = cursor.fetchall()
df = pd.DataFrame(data,columns=['cuisines','total_restaurants'])
print("\nWhich cuisines are most common in Bangalore(quer7)\n",df)

In [ ]:
#query8
query8="""
SELECT cuisines, AVG(rating) as average_rating
FROM Restaurant
GROUP BY cuisines
ORDER BY average_rating DESC
Limit 5;
"""
cursor.execute(query8)
data=cursor.fetchall()
df=pd.DataFrame(data,columns=["cuisines","average_rating"])
print("\n Which cuisines receive the highest average ratings? (query8)\n",df)

In [ ]:
query9="""
SELECT
    cuisines,
    ROUND(AVG(rating),2) AS average_rating,
    COUNT(*) AS total_restaurants
FROM Restaurant
GROUP BY cuisines
HAVING COUNT(*) < 50
ORDER BY average_rating DESC
limit 10;"""
cursor.execute(query9)
data=cursor.fetchall()
df=pd.DataFrame(data,columns=["cuisines","average_rating","total_restaurants"])
print("\n Which cuisines perform well despite having fewer restaurants? (query9)\n",df)

In [ ]:
#query10
query10="""
SELECT
      CASE
          WHEN approx_cost_for_two_people<=500 THEN "LOW"
          WHEN approx_cost_for_two_people<=1000 THEN "MEDIUM"
          WHEN approx_cost_for_two_people<=2000 THEN "HIGH"
          ELSE "PREMIUM"
      END AS price_range,    
      ROUND(AVG(rating),2) as average_rating,
      COUNT(*) AS total_restaurants
FROM Restaurant
GROUP BY price_range
ORDER BY average_rating DESC;"""

cursor.execute(query5)
data = cursor.fetchall()
df = pd.DataFrame(
    data,
    columns=["price_range", "Average_rating","Total_Restaurants"]
)
print("\n What is the relationship between restaurant cost and rating?(query10)\n",df)

In [ ]:
#query11
query11="""
SELECT
    location,
    AVG(rating) AS avg_rating,
    AVG(approx_cost_for_two_people) AS avg_cost,
    COUNT(*) AS total_restaurants
FROM Restaurant
WHERE approx_cost_for_two_people >= 1000
GROUP BY location
HAVING avg_rating>=4
ORDER BY avg_rating DESC ,avg_cost DESC
LIMIT 10;"""
cursor.execute(query11)
data = cursor.fetchall()
df = pd.DataFrame(
    data,
    columns=["location", "avg_rating","avg_cost","Total_Restaurants"])

print("\n Which locations are ideal for premium restaurant onboarding?(query11)\n",df)

In [ ]:
#query12
query12="""
SELECT location,ROUND(AVG(rating),2) AS avg_rating,COUNT(*) AS Total_restaurants
FROM Restaurant
GROUP BY location
HAVING COUNT(*) > 50
ORDER BY avg_rating ASC,Total_restaurants DESC;"""
cursor.execute(query12)
data = cursor.fetchall()
df = pd.DataFrame(
    data,
    columns=["location", "Average_rating","Total_restaurants"]
)
print("\n Which locations show high demand but lower average ratings?(query12)\n",df)

In [ ]:
#query13
query13="""
SELECT online_order,book_table,
ROUND(AVG(rating),2) AS avg_rating
FROM Restaurant
GROUP BY online_order,book_table
ORDER BY avg_rating desc
LIMIT 5;"""
cursor.execute(query13)
data = cursor.fetchall()
df = pd.DataFrame(
    data,
    columns=["online_order","book_table", "Average_rating"]
)
print("\n Do restaurants offering both online ordering and table booking perform better(query13)\n",df)

In [30]:
# Create orders table
cursor.execute("""
CREATE TABLE IF NOT EXISTS orders (
    order_id TEXT,
    restaurant_name TEXT,
    order_date TEXT,
    order_value FLOAT,
    discount_used TEXT,
    payment_method TEXT
    );
""")
cursor.execute("DELETE FROM orders")
data2 = df2.values.tolist()
cursor.executemany("""
INSERT INTO orders (
order_id, restaurant_name, order_date, order_value, discount_used,
payment_method
) VALUES (?, ?, ?, ?, ?, ?)
""", data2)
conn.commit()

In [ ]:
#query21
query21="""
SELECT restaurant_name,count(order_id) as no_of_orders
from orders
GROUP BY restaurant_name
ORDER BY  no_of_orders DESC
limit 10;"""
cursor.execute(query21)
data2 = cursor.fetchall()
df = pd.DataFrame(
    data2,
    columns=["restaurant_name","no_of_orders"]
)
print("\n Which restaurants have the highest number of orders?(query21)\n",df)

In [ ]:
#query22
query22="""
SELECT 
    discount_used,
    COUNT(order_id) AS total_orders,
    ROUND(AVG(order_value),2) AS avg_order_value,
    ROUND(SUM(order_value),2) AS total_revenue
FROM orders
GROUP BY discount_used;"""
cursor.execute(query22)
data2 = cursor.fetchall()
df = pd.DataFrame(
    data2,
    columns=["discount_used","Total_orders","avg_order_value","total_revenue"]
)
print("\n How does discount usage affect total order value?(query22)\n",df)

In [ ]:
#query23
query23="""
SELECT 
    payment_method,
    COUNT(order_id) AS total_orders
    FROM orders
GROUP BY payment_method
ORDER BY total_orders DESC;"""
cursor.execute(query23)
data2 = cursor.fetchall()
df = pd.DataFrame(
    data2,
    columns=["payment_method","total_orders"]
)
print("\n Which payment methods are most commonly used?(query23)\n",df)

In [ ]:
#query24
query24="""
SELECT 
    payment_method,
    COUNT(order_id) AS total_orders,
    ROUND(AVG(order_value),2) AS avg_order_value,
    ROUND(SUM(order_value)) AS total_revenue
FROM orders
GROUP BY payment_method
ORDER BY total_revenue DESC;"""
cursor.execute(query24)
data2 = cursor.fetchall()
df = pd.DataFrame(
    data2,
    columns=["payment_method","total_orders","avg_order_value","total_revenue"]
)
print("\n Does payment method influence order value? (query24)\n",df)

In [ ]:
#query25
query25="""
SELECT
    CASE
        WHEN order_value <= 600 THEN 'Low'
        WHEN order_value <= 1300 THEN 'Medium'
        ELSE 'High'
    END AS order_range,
    COUNT(*) AS total_orders
    FROM orders
    GROUP BY order_range
    ORDER BY total_orders DESC;"""
cursor.execute(query25)
data2 = cursor.fetchall()
df = pd.DataFrame(
    data2,
    columns=["order_range","total_orders"]
)
print("\n What is the most common order value range (low, medium, high)? (query25)\n",df)